# 示例: 运行带汇流节点的河网模型

**脚本:** `examples/run_junction_model.py`

## 目的
此示例用于演示如何使用`SimulationController`和`Junction`组件来构建和运行一个包含两条河流汇合的河网模型。

此笔记本将展示如何：
1.  创建多个独立的河流组件和一个`Junction`（汇流点）组件。
2.  在控制器中将这些组件连接成一个“Y”型的网络拓扑结构。
3.  为不同的上游组件定义不同的输入（边界条件）。
4.  运行模拟并验证在汇流点的水量平衡是否正确。
5.  可视化上游支流和下游干流的流量过程线。

## 1. 环境设置

首先，我们导入所需的框架组件和模型组件。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys
import os

# 将项目根目录添加到Python路径
sys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname('__file__'), '..')))

from common.controller import SimulationController
from preissmann_model.model import HydraulicModel
from preissmann_model.reach import RiverReach
from preissmann_model.cross_section import RectangularCrossSection
from common.junction import Junction

## 2. 创建模型组件

我们创建一个辅助函数 `create_river` 来快速生成河流组件。然后，我们创建三个河流组件（`river_A`, `river_B`, `river_C`）和一个汇流点组件（`J1`）。

In [ ]:
def create_river(name, num_nodes=5, length=500.0, slope=0.001):
    reach_geom = RiverReach(
        cross_sections=[RectangularCrossSection(width=10) for _ in range(num_nodes)],
        lengths=np.full(num_nodes - 1, length / (num_nodes - 1)),
        slope=slope,
        manning_n=0.03
    )
    river = HydraulicModel(
        name=name,
        reach=reach_geom,
        dt=10.0,
        downstream_level=2.0
    )
    river.Q[:] = 0.1
    river.Z[:] = 2.5
    return river

river_A = create_river("river_A")
river_B = create_river("river_B")
junction = Junction(name="J1")
river_C = create_river("river_C", num_nodes=11, length=1000.0)

print("创建组件: river_A, river_B, J1, river_C")

## 3. 构建和连接网络

我们将所有组件添加到控制器中，并定义它们之间的连接关系：`river_A` 和 `river_B` 的出流都汇入 `J1`，而 `J1` 的出流则成为 `river_C` 的入流。

In [ ]:
controller = SimulationController()
controller.add_component(river_A)
controller.add_component(river_B)
controller.add_component(junction)
controller.add_component(river_C)

controller.connect("river_A", "J1")
controller.connect("river_B", "J1")
controller.connect("J1", "river_C")
print("网络连接: (river_A, river_B) -> J1 -> river_C")

## 4. 定义和运行模拟

我们为两条上游支流（`river_A` 和 `river_B`）分别定义不同的入流过程线。然后我们运行模拟，并记录下每个时间步各个组件的出流，以便后续分析和绘图。

In [ ]:
num_steps = 50
dt_controller = 10.0

inflow_A = np.linspace(10, 20, num_steps)
inflow_B = np.linspace(15, 10, num_steps)

global_inputs = {
    'river_A': {'Q_inflow': inflow_A},
    'river_B': {'Q_inflow': inflow_B}
}

history = []
history_generator = controller.run(
    num_steps=num_steps,
    dt=dt_controller,
    global_inputs=global_inputs
)
for status in history_generator:
    current_step_history = {}
    for name, comp in controller.components.items():
        current_step_history[name] = {'outflow': comp.get_outflow(), 'inflow': getattr(comp, 'total_inflow', 0)}
    history.append(current_step_history)

print("\n模拟完成。")

## 5. 验证和可视化结果

首先，我们检查在最后一个时间步，汇入`J1`的总流量是否等于`river_A`和`river_B`的出流量之和，以验证水量平衡。

然后，我们绘制所有关键点的流量过程线。从图中可以看到，`J1`的出流量（黄线）确实是`river_A`（蓝线）和`river_B`（绿线）出流量的和。而`river_C`的出流量（红线）则是`J1`的流量经过河道演算后的结果。

In [ ]:
# 验证
final_outflow_A = history[-1]['river_A']['outflow']
final_outflow_B = history[-1]['river_B']['outflow']
final_inflow_J1 = history[-1]['J1']['inflow']

print(f"最后一个时间步 river_A 的出流: {final_outflow_A:.3f}")
print(f"最后一个时间步 river_B 的出流: {final_outflow_B:.3f}")
print(f"最后一个时间步 J1 的总入流: {final_inflow_J1:.3f}")
assert np.isclose(final_inflow_J1, final_outflow_A + final_outflow_B), "水量平衡错误!"
print("汇流点水量平衡正确。")

# 可视化
outflow_A_hist = [h['river_A']['outflow'] for h in history]
outflow_B_hist = [h['river_B']['outflow'] for h in history]
outflow_J1_hist = [h['J1']['outflow'] for h in history]
outflow_C_hist = [h['river_C']['outflow'] for h in history]
timesteps = np.arange(num_steps) * dt_controller / 60 # minutes

plt.figure(figsize=(12, 7))
plt.plot(timesteps, outflow_A_hist, 'b-^', label='Outflow from river_A')
plt.plot(timesteps, outflow_B_hist, 'g-v', label='Outflow from river_B')
plt.plot(timesteps, outflow_J1_hist, 'y-s', label='Outflow from Junction J1 (Sum)')
plt.plot(timesteps, outflow_C_hist, 'r-o', label='Outflow from river_C (Routed)')

plt.title('Network Simulation with a Junction')
plt.xlabel('Time (minutes)')
plt.ylabel('Flow (m³/s)')
plt.legend()
plt.grid(True)
plt.show()